# 02. BB84 basis sifting과 QBER

목표: BB84의 핵심 흐름과 intercept-resend 공격이 만드는 error를 toy simulation으로 확인한다. Quantum state, error correction, privacy amplification과 composable security proof를 재현하지 않는다.

## 1. 교육용 측정 model

같은 basis로 측정하면 원 bit를 얻고, 다른 basis면 무작위 결과를 얻는다고 단순화한다. `0`은 Z basis, `1`은 X basis다.

In [ ]:
import random


def measure(bit: int, prepared_basis: int, measurement_basis: int, rng: random.Random) -> int:
    if prepared_basis == measurement_basis:
        return bit
    return rng.randrange(2)


rng = random.Random(7)
assert measure(1, 0, 0, rng) == 1

## 2. Alice, Eve와 Bob

Eve가 일부 signal을 임의 basis로 측정하고 그 결과를 다시 보낸다. Bob은 자신의 basis로 측정한다. 이후 Alice와 Bob이 같은 basis를 쓴 위치만 sifted key 후보로 남긴다.

In [ ]:
def run_bb84(signal_count: int, eve_fraction: float, seed: int) -> dict:
    rng = random.Random(seed)
    alice_bits = [rng.randrange(2) for _ in range(signal_count)]
    alice_bases = [rng.randrange(2) for _ in range(signal_count)]
    bob_bases = [rng.randrange(2) for _ in range(signal_count)]
    bob_bits = []

    for bit, alice_basis, bob_basis in zip(alice_bits, alice_bases, bob_bases):
        if rng.random() < eve_fraction:
            eve_basis = rng.randrange(2)
            eve_bit = measure(bit, alice_basis, eve_basis, rng)
            received = measure(eve_bit, eve_basis, bob_basis, rng)
        else:
            received = measure(bit, alice_basis, bob_basis, rng)
        bob_bits.append(received)

    sifted = [
        (a, b)
        for a, b, a_basis, b_basis in zip(alice_bits, bob_bits, alice_bases, bob_bases)
        if a_basis == b_basis
    ]
    mismatches = sum(a != b for a, b in sifted)
    return {
        "signals": signal_count,
        "sifted_bits": len(sifted),
        "qber": mismatches / len(sifted),
    }


baseline = run_bb84(20_000, eve_fraction=0.0, seed=11)
attacked = run_bb84(20_000, eve_fraction=1.0, seed=11)
print("baseline", baseline)
print("full intercept-resend", attacked)
assert baseline["qber"] == 0.0
assert 0.20 < attacked["qber"] < 0.30

## 3. 공격 비율 sweep

Intercept 비율이 커질수록 평균 QBER가 증가한다. 실제 channel noise도 error를 만들므로 observed QBER만으로 Eve의 존재와 원인을 유일하게 식별할 수는 없다.

In [ ]:
results = []
for index, fraction in enumerate((0.0, 0.25, 0.5, 0.75, 1.0)):
    result = run_bb84(50_000, eve_fraction=fraction, seed=100 + index)
    results.append((fraction, result["qber"]))
    print(f"eve_fraction={fraction:.2f}, qber={result['qber']:.4f}")

assert results[-1][1] > results[1][1]

## 4. 빠진 보안 단계

Production QKD에는 authenticated classical channel, finite-key parameter estimation, information reconciliation, privacy amplification, key confirmation과 secure device implementation이 필요하다. 인증이 없으면 이 simulation의 basis 공개 과정 자체가 MITM에 취약하다.